In [1]:
# Save file list - adjust based on climate model and location of files

In [2]:
import os
import json
from datetime import datetime
from utils.utils import get_scenario_config

In [3]:
# === Filename builders for CESM2 ===
def build_arise_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/collections/ARISE-SAI-1.5/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_ssp245_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_hist_path(ens_num, date_range):
    base = (
        "/glade/campaign/cesm/development/wawg/WACCM6-TSMLT-HIST/"
        f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base


def build_g6_path(ens_num, date_range):
    base = (
        "/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/"
        f"b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{ens_num}.cam.h4.O3_SRF.{date_range}.nc"
    )
    return base

In [4]:
# === File list generator for CESM2 ===
def cesm_file_list(scenario, ens_num):
    files = []
    num = f"{ens_num:02d}"

    if scenario == "ARISE":
        if ens_num in [5, 8, 9]:  # These files are saved yearly
            for year in range(2035, 2070 if ens_num in [8, 9] else 2069):
                start = f"{year}010100"
                end = f"{year+1}010100"
                date_range = f"{start}-{end}"
                files.append(build_arise_path(num, date_range))
            if ens_num in [5]:
                files.append(build_arise_path(num, "2069010100-2069123100"))
        else:  # These files are saved in decades
            for start, end in [(2035, 2045), (2045, 2055),
                               (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_arise_path(num, date_range))

    elif scenario == "SSP245":
        if ens_num <= 5:  # saved through 2100
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045),
                               (2045, 2055), (2055, 2065), (2065, 2075)]:
                date_range = f"{start}010100-{end}010100"
                files.append(build_ssp245_path(num, date_range))
        else:  # ends in 20691231
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045),
                               (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_ssp245_path(num, date_range))

    elif scenario == "SSP245_G6":
        for start, end in [(2015, 2025), (2025, 2035), (2035, 2045),
                           (2045, 2055), (2055, 2065), (2065, 2075),
                           (2075, 2085)]:
            date_range = f"{start}010100-{end}010100"
            files.append(build_ssp245_path(num, date_range))

    elif scenario == "hist":
        files.append(build_hist_path(num, "1988010100-1998010100"))
        files.append(build_hist_path(num, "1998010100-1999123100"))
        files.append(build_hist_path(num, "2000010100-2010010100"))
        files.append(build_hist_path(num, "2010010100-2015011500"))

    elif scenario == "G6-1.5K":
        for start, end in [(2035, 2045), (2045, 2055), (2055, 2065),
                           (2065, 2075), (2075, 2084)]:
            date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2075
                    else f"{start}010100-{end}010100"
                )
            files.append(build_g6_path(num, date_range))

    return files

In [9]:
def ukesm_file_list(scneario, ens_num):
    files = []
    file_path = f"/glade/work/awells/air_quality/UKESM1/data/{scenario}/"
    if scenario == "hist":
        files.append(os.path.join(file_path, "sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_199001010030-199912302330.nc"))
        files.append(os.path.join(file_path, "sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_200001010030-200912302330.nc"))
        files.append(os.path.join(file_path, "sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_201001010030-201412302330.nc"))

    else:
        # one file per scenario and ensemble
        files.append(os.path.join(file_path, f"ukesm_{scenario}_o3_3hr_{ens_num:02d}.nc"))

    return files

In [6]:
def get_file_list(model, scenario, ens_num):
    try:
        return MODEL_HANDLERS[model](scenario, ens_num)
    except KeyError:
        raise ValueError(
            f"Model {model} not supported. Options: {list(MODEL_HANDLERS)}"
        )

In [7]:
def save_file_list_with_metadata(model, scenario, ens_num, file_list, DIR, filename):
    data = {
        "model": model,
        "scenario": scenario,
        "ensemble_number": ens_num,
        "generated_on": datetime.now().isoformat(),
        "files": file_list
    }
    file_path = os.path.join(DIR, filename)
    with open(file_path, "w") as f:
        json.dump(data, f, indent=2)

In [10]:
# === Master lookup ===
MODEL_HANDLERS = {
    "CESM2": cesm_file_list,
    "UKESM1": ukesm_file_list,
}

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

SAVE_DIR = f"/glade/work/awells/air_quality/{model}/ozone/file_paths/"

for ens_num in ensemble_members:
    print(f"Building list of files for {scenario}, Ensemble {ens_num:02d}")
    file_list = get_file_list(model, scenario, ens_num)
    save_file_list_with_metadata(
        model,
        scenario,
        ens_num,
        file_list,
        SAVE_DIR,
        f"file_list_{scenario}_{ens_num}.json"
    )

print("All processing complete.")

Building list of files for SSP245_G6, Ensemble 01
Building list of files for SSP245_G6, Ensemble 02
Building list of files for SSP245_G6, Ensemble 03
All processing complete.
